In [ ]:
import numpy as np
import pandas as pd
from data import (
    Vertebral_column,
    Vertebral_column_KF,
    Ecoli_TestSize,
    Jm1_TestSize,
    Jm1_SMOTETomek,
    Haberman_TestSize,
    Transfution_TestSize,
    Pima_TestSize,
    Co_Author_TestSize,
    churn,
    Churn_SMOTETomek_IR,
    Abanole_TestSize,
    Abanole_SMOTETomek,
    Ecoli_SMOTETomek_IR,
    Haberman_SMOTETomek_IR,
    Pima_SMOTETomek_IR,
    Transfution_SMOTETomek_IR,
    Co_Author_SMOTETomek_IR,
    Yeast_TestSize,
    Yeast_SMOTETomek,
    Vertebral_column_SMOTETomek_IR,
    Cm1_TestSize,
    Cm1_SMOTETomek
)
import trainning_of_adaboost as toa
import fearn_adaboost as fearn_toa
from sklearn.ensemble import AdaBoostClassifier
import adaboost_svm, ImAda_DecisionTree
from report import report
from sklearn.metrics import classification_report, precision_recall_fscore_support as score
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, f1_score, precision_score
import math
from datetime import datetime
import csv
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from wsvm.application import Wsvm
from sklearn.svm import SVC

In [ ]:
def compute_metrics(y_test,y_pred):
    cm_WSVM = confusion_matrix(y_test, y_pred)
    se = cm_WSVM[1,1]/(cm_WSVM[1,0]+cm_WSVM[1,1])
    sp = cm_WSVM[0,0]/(cm_WSVM[0,0]+cm_WSVM[0,1])
    gmean = math.sqrt(se*sp)
    f1s = f1_score(y_test,y_pred)
    acc = accuracy_score(y_test,y_pred)
    pre = precision_score(y_test,y_pred)
    auc = roc_auc_score(y_test, y_pred)

    return sp, se, gmean, f1s, pre, acc, auc, cm_WSVM

In [ ]:
# 1. SVM lib
def svm_lib(X_train, y_train,X_test):
    clf = SVC(probability=True, kernel='linear')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred

In [ ]:
# 3. DecisionTree
from sklearn import tree
def decisiontree(X_train, y_train,X_test):
    clf = tree.DecisionTreeClassifier()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred

In [ ]:
# 4. WSVM
def wsvm(C,X_train, y_train,X_test,distribution_weight=None):
    model = Wsvm(C,distribution_weight)
    model.fit(X_train, y_train)
    test_pred = model.predict(X_test)
    return test_pred

In [ ]:
# 5. AdaBoost SVM
def ada_svm(X_train, y_train, X_test):
    clf = AdaBoostClassifier(SVC(probability=True,kernel='linear'),n_estimators=100,learning_rate=1.0, algorithm='SAMME')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred


In [ ]:
# 6. AdaBoost DecisionTree
def ada_decisiontree(X_train, y_train,X_test):
    clf = AdaBoostClassifier(n_estimators=100)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred

In [ ]:
# 7. AdaBoost WSVM
def ada_wsvm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(X_train, y_train, M, C, instance_categorization=True, proposed_preprocessing=False, proposed_alpha=False, test_something=False, theta=theta)
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a

In [ ]:
#10. IM.AdaBoost-12 WSVM
def imada_12_wsvm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(X_train, y_train, M, C, instance_categorization=True, proposed_preprocessing=True, proposed_alpha=True, test_something=False, theta=theta)
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a

In [ ]:
#13. IM.AdaBoost12 + SVM
def imada_12_svm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(X_train, y_train, M, C, instance_categorization=False, proposed_preprocessing=True, proposed_alpha=True, test_something=False, theta=theta)
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a

In [ ]:
def imada_12_decisiontree(M, theta, X_train, y_train, X_test):
    clf, a = ImAda_DecisionTree.fit(
        X_train, y_train, M, proposed_preprocessing=True, proposed_alpha=True, theta=theta
    )
    y_pred = ImAda_DecisionTree.predict(X_test, a, clf)
    return y_pred, a


# EANR-AdaBoost: thay ca init + confident
def eanr_adaboost_wsvm(M, C, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=True,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        use_entropy_init=True,
        use_noise_robust_confident=True,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost_svm(M, C, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=False,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        use_entropy_init=True,
        use_noise_robust_confident=True,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost_decisiontree(M, X_train, y_train, X_test):
    clf, a = ImAda_DecisionTree.fit(
        X_train, y_train, M,
        proposed_preprocessing=True,
        proposed_alpha=True,
        use_entropy_init=True,
        use_noise_robust_confident=True,
    )
    y_pred = ImAda_DecisionTree.predict(X_test, a, clf)
    return y_pred, a


# Kich ban 1: SoftMargin_EARN_AdaBoost (de xuat 2 + 3 + 4)
def softmargin_earn_adaboost_wsvm(M, C, X_train, y_train, X_test):
    w, b, a = fearn_toa.fit(
        X_train, y_train, M, C,
        instance_categorization=True,
        proposed_preprocessing=True,
        test_something=False,
        use_entropy_init=True,
        use_fuzzy_spatial_weight=False,
    )
    y_pred = fearn_toa.predict(X_test, w, b, a, M)
    return y_pred, a


def softmargin_earn_adaboost_svm(M, C, X_train, y_train, X_test):
    w, b, a = fearn_toa.fit(
        X_train, y_train, M, C,
        instance_categorization=False,
        proposed_preprocessing=True,
        test_something=False,
        use_entropy_init=True,
        use_fuzzy_spatial_weight=False,
    )
    y_pred = fearn_toa.predict(X_test, w, b, a, M)
    return y_pred, a


# Kich ban 2: FEARN_AdaBoost (de xuat 1 + 2 + 3 + 4)
def fearn_adaboost_wsvm(M, C, X_train, y_train, X_test):
    w, b, a = fearn_toa.fit(
        X_train, y_train, M, C,
        instance_categorization=True,
        proposed_preprocessing=True,
        test_something=False,
        use_entropy_init=True,
        use_fuzzy_spatial_weight=True,
    )
    y_pred = fearn_toa.predict(X_test, w, b, a, M)
    return y_pred, a


def fearn_adaboost_svm(M, C, X_train, y_train, X_test):
    w, b, a = fearn_toa.fit(
        X_train, y_train, M, C,
        instance_categorization=False,
        proposed_preprocessing=True,
        test_something=False,
        use_entropy_init=True,
        use_fuzzy_spatial_weight=True,
    )
    y_pred = fearn_toa.predict(X_test, w, b, a, M)
    return y_pred, a

In [ ]:
####################################### TEST SIZE SCRIPT - FIND BEST PARAMETERS ################################
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import threading

# -----------------------------
# 1) Cau hinh tham so chay
# -----------------------------
M = [10, 15, 20, 25]
C = [10, 100, 1000]
theta = [0.3, 0.5, 0.7, 1, 1.5, 2]
N = 1
test_size = [0.2]

dataset_name = "jm1"
dataset_original = Jm1_TestSize
dataset_smote = Jm1_SMOTETomek

new_rate = None

time = datetime.now().strftime("%d%m%Y_%H%M%S")
filepath = f'./Experiment/Data_{dataset_name}_{time}_TestSize.csv'
max_workers = min(12, (os.cpu_count() or 6))

# -----------------------------
# 2) Bat/tat method tai day
# True = chay, False = bo qua
# -----------------------------
METHOD_SWITCHES = {
    # Nhom co ban (khong dung M/C/theta)
    "Decision Tree": False,
    "SVM (lib)": False,
    "ADA_DSTree": False,
    "ADA_SVM": False,

    # Nhom dung theta
    "WSVM": False,
    "ADA_WSVM": False,
    "ImADA_12_DecisionTree": False,
    "ImADA_12_SVM": False,
    "ImADA_12_WSVM": False,

    # Nhom KHONG dung theta
    "EANR-AdaBoost_DecisionTree": False,
    "EANR-AdaBoost_SVM": False,
    "EANR-AdaBoost_WSVM": False,
    "SoftMargin_EARN_AdaBoost_SVM": True,
    "SoftMargin_EARN_AdaBoost_WSVM": True,
    "FEARN_AdaBoost_SVM": True,
    "FEARN_AdaBoost_WSVM": True,
}

header = [
    'Test Size', 'Method', 'M', 'C', 'theta', 'SP', 'SE', 'Gmean',
    'F1 Score', 'Precision', 'Accuracy', 'AUC', 'Ma tran nham lan',
    'List of err_w', 'List of alpha'
]
file_lock = threading.Lock()

with open(filepath, 'a', encoding='UTF8', newline='') as f1:
    writer = csv.writer(f1)
    writer.writerow(header)


def is_enabled(method_name):
    return METHOD_SWITCHES.get(method_name, False)


def append_row(row):
    with file_lock:
        with open(filepath, 'a', encoding='UTF8', newline='') as f1:
            writer = csv.writer(f1)
            writer.writerow(row)


def load_data_flexible(dataset_module, testsize, new_rate_val=None):
    if new_rate_val is None:
        return dataset_module.load_data(test_size=testsize)
    try:
        return dataset_module.load_data(test_size=testsize, new_rate=new_rate_val)
    except TypeError:
        return dataset_module.load_data(test_size=testsize)


def safe_run_and_append(method_name, testsize, m, c, t, y_test, run_fn):
    try:
        y_pred, alpha = run_fn()
        sp, se, gmean, f1s, pre, acc, auc, cm = compute_metrics(y_test, y_pred)
        append_row([testsize, method_name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm), 'None', alpha])
    except Exception as ex:
        append_row([testsize, method_name, m, c, t, 'ERR', 'ERR', 'ERR', 'ERR', 'ERR', 'ERR', 'ERR', str(ex), 'None', 'None'])


def submit_if_enabled(executor, futures, method_key, variant_tag, testsize, m, c, t, y_test, run_fn):
    if not is_enabled(method_key):
        return
    futures.append(executor.submit(
        safe_run_and_append,
        f"{method_key} | {variant_tag}",
        testsize,
        m,
        c,
        t,
        y_test,
        run_fn,
    ))


dataset_variants = [
    ("ORIG", dataset_original),
    ("SMOTE", dataset_smote)
]

for n in range(0, N):
    print("Lan boc:", n + 1)
    for testsize in test_size:
        for variant_tag, dataset_module in dataset_variants:
            print(f"Dataset variant: {variant_tag}")
            X_train, y_train, X_test, y_test = load_data_flexible(
                dataset_module, testsize, new_rate_val=new_rate
            )
            num_samples, _ = X_train.shape
            distribution_weight = np.ones(num_samples)

            futures = []
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                # Nhom co ban
                submit_if_enabled(
                    executor, futures, "Decision Tree", variant_tag, testsize,
                    "None", "none", "none", y_test,
                    lambda: (decisiontree(X_train, y_train, X_test), "None")
                )
                submit_if_enabled(
                    executor, futures, "SVM (lib)", variant_tag, testsize,
                    "None", "none", "none", y_test,
                    lambda: (svm_lib(X_train, y_train, X_test), "None")
                )
                submit_if_enabled(
                    executor, futures, "ADA_DSTree", variant_tag, testsize,
                    "None", "none", "none", y_test,
                    lambda: (ada_decisiontree(X_train, y_train, X_test), "None")
                )
                submit_if_enabled(
                    executor, futures, "ADA_SVM", variant_tag, testsize,
                    "None", "none", "none", y_test,
                    lambda: (ada_svm(X_train, y_train, X_test), "None")
                )

                # Nhom dung M, C
                for m in M:
                    for c in C:
                        # Nhom co su dung theta
                        # for t in theta:
                        #     print(variant_tag, m, c, t)
                        #     submit_if_enabled(
                        #         executor, futures, "WSVM", variant_tag, testsize,
                        #         m, c, t, y_test,
                        #         lambda m=m, c=c: (wsvm(c, X_train, y_train, X_test, distribution_weight), "None")
                        #     )
                        #     submit_if_enabled(
                        #         executor, futures, "ADA_WSVM", variant_tag, testsize,
                        #         m, c, t, y_test,
                        #         lambda m=m, c=c, t=t: ada_wsvm(m, c, t, X_train, y_train, X_test)
                        #     )
                        #     submit_if_enabled(
                        #         executor, futures, "ImADA_12_DecisionTree", variant_tag, testsize,
                        #         m, c, t, y_test,
                        #         lambda m=m, t=t: imada_12_decisiontree(m, t, X_train, y_train, X_test)
                        #     )
                        #     submit_if_enabled(
                        #         executor, futures, "ImADA_12_SVM", variant_tag, testsize,
                        #         m, c, t, y_test,
                        #         lambda m=m, c=c, t=t: imada_12_svm(m, c, t, X_train, y_train, X_test)
                        #     )
                        #     submit_if_enabled(
                        #         executor, futures, "ImADA_12_WSVM", variant_tag, testsize,
                        #         m, c, t, y_test,
                        #         lambda m=m, c=c, t=t: imada_12_wsvm(m, c, t, X_train, y_train, X_test)
                        #     )

                        # EANR-AdaBoost, SoftMargin_EARN_AdaBoost va FEARN_AdaBoost khong dung theta
                        submit_if_enabled(
                            executor, futures, "EANR-AdaBoost_DecisionTree", variant_tag, testsize,
                            m, c, "None", y_test,
                            lambda m=m: eanr_adaboost_decisiontree(m, X_train, y_train, X_test)
                        )
                        submit_if_enabled(
                            executor, futures, "EANR-AdaBoost_SVM", variant_tag, testsize,
                            m, c, "None", y_test,
                            lambda m=m, c=c: eanr_adaboost_svm(m, c, X_train, y_train, X_test)
                        )
                        submit_if_enabled(
                            executor, futures, "EANR-AdaBoost_WSVM", variant_tag, testsize,
                            m, c, "None", y_test,
                            lambda m=m, c=c: eanr_adaboost_wsvm(m, c, X_train, y_train, X_test)
                        )
                        submit_if_enabled(
                            executor, futures, "SoftMargin_EARN_AdaBoost_SVM", variant_tag, testsize,
                            m, c, "None", y_test,
                            lambda m=m, c=c: softmargin_earn_adaboost_svm(m, c, X_train, y_train, X_test)
                        )
                        submit_if_enabled(
                            executor, futures, "SoftMargin_EARN_AdaBoost_WSVM", variant_tag, testsize,
                            m, c, "None", y_test,
                            lambda m=m, c=c: softmargin_earn_adaboost_wsvm(m, c, X_train, y_train, X_test)
                        )
                        submit_if_enabled(
                            executor, futures, "FEARN_AdaBoost_SVM", variant_tag, testsize,
                            m, c, "None", y_test,
                            lambda m=m, c=c: fearn_adaboost_svm(m, c, X_train, y_train, X_test)
                        )
                        submit_if_enabled(
                            executor, futures, "FEARN_AdaBoost_WSVM", variant_tag, testsize,
                            m, c, "None", y_test,
                            lambda m=m, c=c: fearn_adaboost_wsvm(m, c, X_train, y_train, X_test)
                        )

                for future in as_completed(futures):
                    future.result()

print(f"Hoan tat. Da luu ket qua tai: {filepath}")